In [ ]:
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   ${budget.total_limit}")
print(f"  Session token: {budget.primary_token.session_token[:20]}...")

# Extract session token from tokens array (one entry per dimension)
tokens = {t.category: t.session_token for t in budget.tokens}
session_token = tokens["default"]  # simple budget uses "default" category

## Velocity controls — stopping runaway loops

`velocity_max_per_minute` caps the number of `authorize` calls in any rolling 60-second window. This catches retry loops before they accumulate meaningful spend — even if the budget is already exhausted and every call returns `BUDGET_EXHAUSTED`, the velocity counter still increments on each attempt.

Set it on `create_budget` alongside your other safety controls. It can also be updated later via `PATCH /budgets/{id}` without recreating the budget.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

# velocity_max_per_minute=5 means at most 5 authorize calls per 60-second rolling window.
# The 6th call in the same window returns VELOCITY_LIMIT_EXCEEDED immediately —
# before checking the remaining balance. This stops retry loops even when the
# budget is exhausted (calls keep incrementing the counter regardless of approval).
budget_with_velocity = client.create_budget(
    user_id="agent_001",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session — velocity controls demo",
    velocity_max_per_minute=5,  # stop runaway loops: at most 5 calls/min
)

print(f"✓ Budget created: {budget_with_velocity.id}")
print(f"  velocity_max_per_minute=5")
print(f"  First 5 calls/min: evaluated normally (AUTHORIZED or denied on budget rules)")
print(f"  6th call in the same 60-second window: VELOCITY_LIMIT_EXCEEDED")
print(f"  First violation fires the VELOCITY_LIMIT_EXCEEDED webhook once.")
print(f"  Subsequent violations in the same window are silently denied (no new ledger entry).")

In [ ]:
# Authorized transaction
auth = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="JetBlue SFO→JFK roundtrip",
    requested_quantity=270.00,
    idempotency_key="booking-001",
)
print(f"Transaction 1: {auth.decision} — ${auth.approved_quantity}")

client.confirm_event(auth.event_id, confirmed_quantity=267.00)
print("✓ Confirmed. Ledger updated.")

# Denied — over remaining budget
auth2 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Business class upgrade",
    requested_quantity=890.00,
    idempotency_key="booking-002",
)
print(f"Transaction 2: {auth2.decision} — {auth2.denial_reason}")
print("Nothing moved. Ledger recorded the denial.")

print(f"\n→ See the spend tree: https://figuard-sandbox-1.onrender.com/ui")